# Day 21 — Time series with pandas
Objectives:
- Date parsing and DateTimeIndex.
- Resampling (D/W/M), rolling windows.
- Time zone awareness basics.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-21`. Read
`python/ds-60day/companion-guides/day21_time_series_pandas.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

A timestamp represents a point or label in time; a duration represents
an elapsed amount. Parsing text is only the first step: decide whether
timestamps are timezone-naive or timezone-aware and what business zone
controls calendar boundaries. Localizing assigns a zone to wall-clock
values; converting changes the representation of an already-known
instant.

Resampling groups a time index into calendar bins and aggregates each
bin. Rolling windows calculate over neighboring observations or elapsed
time. `shift` aligns earlier values with later rows and is central to
lag features. Sort the time index, make duplicate timestamps and missing
intervals explicit, and document boundary/label rules.

### Vocabulary

- **timestamp:** a date-time value, optionally tied to a timezone.
- **timezone-aware:** carrying an offset/zone interpretation for an instant.
- **localize:** assign a timezone interpretation to naive wall-clock values.
- **convert:** represent an aware instant in another timezone.
- **resample:** group observations into regular time bins.
- **rolling window:** a calculation over neighboring rows or an elapsed interval.

## Syntax anatomy

`series.resample("W-FRI").sum()` requires a datetime-like index, groups
into weeks labeled/ending Friday, and sums each bin. `rolling(3,
min_periods=1).mean()` uses up to three consecutive rows and permits a
result at the beginning. `shift(1)` moves values one row later without
changing the index.

### Worked example 1 — Parse, index, and resample deterministic daily values

Choose and name the calendar boundary. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
import pandas as pd

daily = pd.Series(
    [2, 3, 5, 7, 11],
    index=pd.date_range("2025-01-01", periods=5, freq="D"),
    name="sales",
)
weekly = daily.resample("W-SUN").sum()
(daily.sum(), weekly.to_dict())

**Expected observation:** Both weekly bins together sum to `28`, preserving the additive total. The keys are Sunday-labeled timestamps.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Compare rolling and lagged values

A rolling statistic includes nearby history; a lag exposes a prior observation. Predict first; then run the next cell.

In [ ]:
time_features = pd.DataFrame({
    "sales": daily,
    "rolling_3": daily.rolling(3, min_periods=1).mean(),
    "previous_day": daily.shift(1),
})
time_features.round(2)

**Expected observation:** The first rolling mean is `2.0`; the first previous-day value is missing because no earlier row exists.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Inspect dtype, timezone, sort order, duplicates, minimum/maximum, and inferred frequency.
2. Never compare or combine naive and aware timestamps without an explicit policy.
3. For rolling results, list the exact source rows contributing at boundaries.
4. For resampling, document frequency alias, closed side, label, and aggregation semantics.

**Alternative to compare:** Use row-count windows for evenly sampled observations and time-offset windows when elapsed time—not row count—defines the question.

**Boundary to test:** Daylight-saving ambiguity/nonexistence, duplicate times, irregular gaps, empty bins, and partial boundary windows need policy.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
import pandas as pd, numpy as np
rng = pd.date_range('2024-01-01', periods=120, freq='D')
ts = pd.Series(np.random.randn(len(rng)).cumsum(), index=rng, name='value')
ts.head(), ts.index

# Resample monthly
monthly = ts.resample('ME').mean()
rolling = ts.rolling(window=7).mean()
monthly.head(), rolling.head()


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Parse the supplied timestamp strings, reject unparseable values, localize according to a written source-timezone policy, and convert the result to UTC.
   **Expected behavior:** the final dtype is timezone-aware UTC and invalid text is reported rather than silently dropped.
   **Verify:** inspect the earliest/latest instant and demonstrate that conversion preserves the instant.

2. Using deterministic daily sales, compute a three-observation rolling mean and weekly `W-FRI` sums. **Constraints:** sort the index, choose `min_periods`, and document weekly labels/boundaries.
   **Verify:** trace the first three rolling windows by hand and assert weekly sums preserve the grand total.

### Additional mastery practice

Keep timestamps timezone-aware at system boundaries, declare resampling windows, and reconcile totals whenever frequency changes.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

3. **Prediction:** Predict the difference between `tz_localize` and `tz_convert`, and which one applies to a naive timestamp.
   **Progressive hint:** Localization assigns meaning; conversion changes representation of an instant.
   **Verify:** Assert localizing a naive timestamp creates an aware value and converting it to UTC preserves the same instant; show the inappropriate operation raises.
4. **Tracing:** Trace a three-day rolling mean with `min_periods=1` and identify which observations contribute at the beginning.
   **Progressive hint:** Early windows contain fewer rows when the minimum allows it.
   **Verify:** List source dates contributing to the first three outputs and assert their means for `min_periods=1` exactly.
5. **Implementation:** Resample deterministic daily sales to `W-FRI`, then assert the grand total is preserved.
   **Progressive hint:** Document the weekly label/boundary and use sums for additive measures.
   **Verify:** Assert weekly bin labels follow `W-FRI`, every source date belongs to one bin, and weekly sums equal the daily grand total.
6. **Debugging:** Repair a comparison between naive and timezone-aware timestamps.
   **Progressive hint:** Normalize both sides to an aware UTC contract.
   **Verify:** Show the naive/aware comparison failure, normalize both to aware UTC, and assert chronological comparison now reflects the intended instants.
7. **Edge case and explanation:** Handle an ambiguous or nonexistent daylight-saving local time and explain why silently guessing may corrupt event order.
   **Progressive hint:** Use explicit `ambiguous`/`nonexistent` policy or reject the input.
   **Verify:** Use explicit ambiguous/nonexistent policies on documented DST fixtures and assert the chosen raise/resolve outcome without silently reordering events.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Parse the supplied timestamp strings, reject unparseable values, localize according to a written source-timezone policy, and convert the result to UTC. **Expected behavior:** the final dtype is timezone-aware UTC and invalid text is reported rather than silently dropped. **Verify:** inspect the earliest/latest instant and demonstrate that conversion preserves the instant.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Parse the supplied timestamp strings, reject unparseable values, localize according to a written source-timezone policy, and convert the result to UTC. the final dtype is timezo...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Using deterministic daily sales, compute a three-observation rolling mean and weekly `W-FRI` sums. **Constraints:** sort the index, choose `min_periods`, and document weekly labels/boundaries. **Verify:** trace the first three rolling windows by hand and assert weekly sums preserve the grand total.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Using deterministic daily sales, compute a three-observation rolling mean and weekly `W-FRI` sums. sort the index, choose `min_periods`, and document weekly labels/boundaries. t...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict the difference between `tz_localize` and `tz_convert`, and which one applies to a naive timestamp. **Progressive hint:** Localization assigns meaning; conversion changes representation of an instant. **Verify:** Assert localizing a naive timestamp creates an aware value and converting it to UTC preserves the same instant; show the inappropriate operation raises.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Predict the difference between `tz_localize` and `tz_convert`, and which one applies to a naive timestamp. Localization assigns meaning; conversion changes representation of an...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace a three-day rolling mean with `min_periods=1` and identify which observations contribute at the beginning. **Progressive hint:** Early windows contain fewer rows when the minimum allows it. **Verify:** List source dates contributing to the first three outputs and assert their means for `min_periods=1` exactly.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Trace a three-day rolling mean with `min_periods=1` and identify which observations contribute at the beginning. Early windows contain fewer rows when the minimum allows it. Lis...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Resample deterministic daily sales to `W-FRI`, then assert the grand total is preserved. **Progressive hint:** Document the weekly label/boundary and use sums for additive measures. **Verify:** Assert weekly bin labels follow `W-FRI`, every source date belongs to one bin, and weekly sums equal the daily grand total.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Resample deterministic daily sales to `W-FRI`, then assert the grand total is preserved. Document the weekly label/boundary and use sums for additive measures. Assert weekly bin...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Repair a comparison between naive and timezone-aware timestamps. **Progressive hint:** Normalize both sides to an aware UTC contract. **Verify:** Show the naive/aware comparison failure, normalize both to aware UTC, and assert chronological comparison now reflects the intended instants.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Repair a comparison between naive and timezone-aware timestamps. Normalize both sides to an aware UTC contract. Show the naive/aware comparison failure, normalize both to aware...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Handle an ambiguous or nonexistent daylight-saving local time and explain why silently guessing may corrupt event order. **Progressive hint:** Use explicit `ambiguous`/`nonexistent` policy or reject the input. **Verify:** Use explicit ambiguous/nonexistent policies on documented DST fixtures and assert the chosen raise/resolve outcome without silently reordering events.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Handle an ambiguous or nonexistent daylight-saving local time and explain why silently guessing may corrupt event order. Use explicit `ambiguous`/`nonexistent` policy or reject...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
